In [ ]:
import hashlib
import json
import time


# ================= VOTER CLASS =================
class Voter:

    def __init__(self, voter_id, name):
        self.voter_id = voter_id
        self.name = name
        self.has_voted = False


# ================= CANDIDATE CLASS =================
class Candidate:

    def __init__(self, candidate_id, name):
        self.candidate_id = candidate_id
        self.name = name


# ================= BLOCK CLASS =================
class Block:

    def __init__(self, index, transaction, previous_hash):
        self.index = index
        self.timestamp = time.time()
        self.transaction = transaction
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_data = {
            "index": self.index,
            "timestamp": self.timestamp,
            "transaction": self.transaction,
            "previous_hash": self.previous_hash,
        }

        encoded_data = json.dumps(block_data, sort_keys=True).encode("utf-8")
        return hashlib.sha256(encoded_data).hexdigest()


# ================= BLOCKCHAIN CLASS =================
class VotingBlockchain:

    def __init__(self):
        self.voters = {}
        self.candidates = {}

        genesis_transaction = {
            "type": "GENESIS",
            "message": "First block of Voting Blockchain",
        }

        self.chain = [Block(0, genesis_transaction, "0")]

    # Add Candidate
    def add_candidate(self, candidate_id, name):
        if not candidate_id or not name:
            print("Error: Candidate ID and name cannot be empty.")
            return False

        if candidate_id in self.candidates:
            print("Error: Candidate already exists.")
            return False

        self.candidates[candidate_id] = Candidate(candidate_id, name)
        print(f"Candidate added successfully: {name}")
        return True

    # Add Voter
    def add_voter(self, voter_id, name):
        if not voter_id or not name:
            print("Error: Voter ID and name cannot be empty.")
            return False

        if voter_id in self.voters:
            print("Error: Voter already exists.")
            return False

        self.voters[voter_id] = Voter(voter_id, name)
        print(f"Voter registered successfully: {name}")
        return True

    # Cast Vote
    def cast_vote(self, voter_id, candidate_id):
        voter = self.voters.get(voter_id)
        candidate = self.candidates.get(candidate_id)

        if voter is None:
            print("Error: Voter not found.")
            return False

        if candidate is None:
            print("Error: Candidate not found.")
            return False

        if voter.has_voted:
            print("Error: Voter has already voted.")
            return False

        transaction = {
            "type": "VOTE",
            "voter_id": voter.voter_id,
            "voter_name": voter.name,
            "candidate_id": candidate.candidate_id,
            "candidate_name": candidate.name,
        }

        previous_block = self.chain[-1]

        new_block = Block(len(self.chain), transaction, previous_block.hash)

        self.chain.append(new_block)
        voter.has_voted = True

        print(f"Vote cast successfully for {candidate.name} by {voter.name}!")
        return True

    # Display Blockchain
    def print_blockchain(self):
        print("\n========== BLOCKCHAIN CONTENTS ==========")

        for block in self.chain:
            print("\n" + "-" * 50)
            print("Block Index:", block.index)
            print("Timestamp:", time.ctime(block.timestamp))
            print("Transaction:")
            print(json.dumps(block.transaction, indent=4))
            print("Previous Hash:", block.previous_hash)
            print("Block Hash:", block.hash)

        print("-" * 50)

    # Validate Blockchain
    def validate_chain(self):
        if not self.chain:
            return False

        genesis = self.chain[0]

        if genesis.index != 0:
            return False

        if genesis.previous_hash != "0":
            return False

        if genesis.hash != genesis.calculate_hash():
            return False

        seen_voters = set()

        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            # Verify current block hash
            if current.hash != current.calculate_hash():
                return False

            # Verify previous block connection
            if current.previous_hash != previous.hash:
                return False

            transaction = current.transaction

            if transaction.get("type") != "VOTE":
                return False

            voter_id = transaction.get("voter_id")
            candidate_id = transaction.get("candidate_id")

            if not voter_id or voter_id in seen_voters:
                return False

            if candidate_id not in self.candidates:
                return False

            seen_voters.add(voter_id)

        return True

    # Count Votes
    def count_votes(self):
        results = {candidate_id: 0 for candidate_id in self.candidates}

        for block in self.chain[1:]:
            transaction = block.transaction

            if transaction.get("type") == "VOTE":
                candidate_id = transaction.get("candidate_id")

                if candidate_id in results:
                    results[candidate_id] += 1

        return results

    # Display Voting Results
    def display_results(self):
        print("\n========== VOTING RESULTS ==========")
        results = self.count_votes()

        if not results:
            print("No candidates registered.")
            return

        for candidate_id, count in results.items():
            candidate = self.candidates[candidate_id]
            print(
                f"Candidate: {candidate.name} | ID: {candidate_id} | Votes: {count}"
            )

        print("Total Votes:", sum(results.values()))

    # Display Registered Voters
    def display_voters(self):
        print("\n========== REGISTERED VOTERS ==========")

        if not self.voters:
            print("No voters registered.")
            return

        for voter in self.voters.values():
            status = "Voted" if voter.has_voted else "Not Voted"
            print(
                f"ID: {voter.voter_id} | Name: {voter.name} | Status: {status}"
            )


# ================= MAIN MENU =================
def main():
    voting_system = VotingBlockchain()

    while True:
        print("\n========================================")
        print("       BLOCKCHAIN VOTING SYSTEM")
        print("========================================")
        print("1. Add Candidate")
        print("2. Add Voter")
        print("3. Cast Vote")
        print("4. Display Blockchain")
        print("5. Validate Blockchain")
        print("6. Count Votes")
        print("7. Display Registered Voters")
        print("8. Exit")
        print("========================================")

        choice = input("Enter your choice (1-8): ").strip()

        if choice == "1":
            candidate_id = input("Enter Candidate ID: ").strip()
            name = input("Enter Candidate Name: ").strip()
            voting_system.add_candidate(candidate_id, name)

        elif choice == "2":
            voter_id = input("Enter Voter ID: ").strip()
            name = input("Enter Voter Name: ").strip()
            voting_system.add_voter(voter_id, name)

        elif choice == "3":
            voter_id = input("Enter Voter ID: ").strip()
            candidate_id = input("Enter Candidate ID: ").strip()
            voting_system.cast_vote(voter_id, candidate_id)

        elif choice == "4":
            voting_system.print_blockchain()

        elif choice == "5":
            if voting_system.validate_chain():
                print("\nBlockchain is valid. No tampering detected.")
            else:
                print("\nBlockchain is invalid or has been tampered with.")

        elif choice == "6":
            voting_system.display_results()

        elif choice == "7":
            voting_system.display_voters()

        elif choice == "8":
            print("\nThank you for using Blockchain Voting System!")
            break

        else:
            print("Invalid choice! Please enter a number between 1 and 8.")


# ================= PROGRAM EXECUTION =================
if __name__ == "__main__":
    main()


       BLOCKCHAIN VOTING SYSTEM
1. Add Candidate
2. Add Voter
3. Cast Vote
4. Display Blockchain
5. Validate Blockchain
6. Count Votes
7. Display Registered Voters
8. Exit
Enter your choice (1-8): 1
Enter Candidate ID: CD01
Enter Candidate Name: AARUSH KUMAR
Candidate added successfully: AARUSH KUMAR

       BLOCKCHAIN VOTING SYSTEM
1. Add Candidate
2. Add Voter
3. Cast Vote
4. Display Blockchain
5. Validate Blockchain
6. Count Votes
7. Display Registered Voters
8. Exit
Enter your choice (1-8): 2
Enter Voter ID: V01
Enter Voter Name: BIREN KUMAR
Voter registered successfully: BIREN KUMAR

       BLOCKCHAIN VOTING SYSTEM
1. Add Candidate
2. Add Voter
3. Cast Vote
4. Display Blockchain
5. Validate Blockchain
6. Count Votes
7. Display Registered Voters
8. Exit
Enter your choice (1-8): 3
Enter Voter ID: V01
Enter Candidate ID: CD01
Vote cast successfully for AARUSH KUMAR by BIREN KUMAR!

       BLOCKCHAIN VOTING SYSTEM
1. Add Candidate
2. Add Voter
3. Cast Vote
4. Display Blockchain
5. Va